# GraphRAG : Système de Recherche Augmentée par Génération combinant Bases Vectorielles et Graphes

## 📋 Vue d'ensemble

Ce notebook présente une implémentation avancée d'un système **RAG (Retrieval-Augmented Generation)** qui combine deux approches complémentaires pour répondre à des questions complexes sur des documents juridiques algériens :

### 🎯 Objectif Principal
Créer un système intelligent capable de répondre à des questions sur des textes législatifs et réglementaires en exploitant :
- **La recherche sémantique** (base vectorielle ChromaDB) pour trouver des passages pertinents dans les documents
- **Les relations structurées** (base de graphe Neo4j) pour identifier les liens entre documents (modifications, citations, visas)

### 🔧 Composants du Système

1. **Base de données vectorielle (ChromaDB)** : Stocke les documents sous forme d'embeddings pour la recherche sémantique
2. **Base de données graphe (Neo4j)** : Modélise les relations entre documents juridiques
3. **Trois modes de requête** :
   - RAG Vectoriel seul : Questions sur le contenu des documents
   - RAG Graphique seul : Questions sur les relations entre documents
   - RAG Combiné : Exploite les deux sources pour des réponses complètes

### 📚 Cas d'usage
- Recherche d'informations dans les constitutions algériennes (1996-2020)
- Identification des documents qui modifient d'autres documents
- Analyse des articles contenus dans des lois spécifiques
- Traçabilité des références et citations entre textes juridiques

---

## 📑 Table des Matières

1. **Configuration de l'environnement** : Chargement des clés API et configuration des logs
2. **Configuration centralisée** : Définition des modèles et paramètres
3. **Préparation des données** : Chargement et vectorisation des documents markdown
4. **Initialisation des bases de données** : Connexion à Neo4j et ChromaDB
5. **Définition des chaînes RAG** :
   - Chaîne RAG Vectorielle
   - Chaîne RAG Graphique (avec requêtes Cypher)
   - Chaîne RAG Combinée
6. **Démonstrations** : 10 exemples de requêtes couvrant différents scénarios

---

## 🔧 Étape 1 : Configuration de l'Environnement

### Objectif
Cette section initialise l'environnement de travail en chargeant les clés API nécessaires et en configurant le système de logging.

### Ce qui se passe ici :
1. **Chargement des variables d'environnement** : Utilisation de `python-dotenv` pour charger les clés API depuis un fichier `.env`
2. **Configuration du logging** : Désactivation des logs verbeux de LangChain pour un affichage plus propre
3. **Récupération de la clé GROQ** : Extraction de la clé API GROQ pour utiliser les modèles de langage

### Variables importantes :
- `groq_key` : Clé API pour accéder aux modèles de langage Groq (LLM)

In [32]:
# Cell 1: Environment Setup
# Import necessary libraries for environment variable management and logging control
from dotenv import load_dotenv, find_dotenv
import os
from langchain.globals import set_verbose, set_debug

# Disable debug and verbose logging for cleaner output
set_debug(False)
set_verbose(False)

# Robustly locate and load the nearest .env file (starting from CWD)
dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Loaded .env from: {dotenv_path}")
else:
    # Fallback to default behavior
    load_dotenv(override=True)
    print("No explicit .env found via find_dotenv. Attempted default load.")

# Set the GROQ_API_KEY environment variable from the loaded variables for API access
groq_key = os.getenv("GROQ_API_KEY", "")

Loaded .env from: d:\Documents\py\projects\graph_rag\.env


## ⚙️ Étape 2 : Configuration Centralisée

### Objectif
Définir tous les paramètres et noms de modèles dans un seul endroit pour faciliter la maintenance et les modifications.

### Ce qui se passe ici :
1. **Modèles d'embeddings** : Spécification du modèle HuggingFace pour convertir le texte en vecteurs
2. **Modèles LLM** : Configuration des différents modèles de langage pour chaque tâche :
   - RAG vectoriel : modèle rapide pour répondre aux questions de contenu
   - RAG graphique : modèle puissant pour générer des requêtes Cypher
   - RAG combiné : modèle pour synthétiser les résultats
3. **Chemins de fichiers** : Définition des répertoires pour les données et la base vectorielle
4. **Paramètres de base de données** : Nom de la collection ChromaDB

### Avantages de cette approche :
- Changement facile des modèles en un seul endroit
- Maintenance simplifiée
- Configuration claire et lisible

In [ ]:
# --- Cell 2: Centralized Configuration ---
# Define all model names, paths, and other constants here for easy management.

# Model Names
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L12-v2"
VECTOR_RAG_LLM = "llama-3.1-8b-instant"
GRAPH_RAG_LLM = "openai/gpt-oss-120b"  # For Cypher generation
COMBINED_RAG_LLM = "llama-3.1-8b-instant"
GRAPH_RESPONSE_LLM = "openai/gpt-oss-120b"  # For summarizing graph results

# File Paths
PERSIST_DIR = "./chroma_db"
DATA_DIR = "data"

# Vector DB Settings
COLLECTION_NAME = "data_base"

## 📚 Étape 3 : Préparation des Données

### Objectif
Charger les documents juridiques et les préparer pour la recherche sémantique en les transformant en vecteurs.

### Ce qui se passe ici :

#### 1. Chargement des Documents
- **DirectoryLoader** : Charge tous les fichiers Markdown depuis le dossier `data/`
- Les documents incluent les constitutions algériennes et divers textes législatifs

#### 2. Découpage Structuré
- **MarkdownHeaderTextSplitter** : Découpe les documents selon leurs titres (H1, H2, H3)
- Préserve la structure hiérarchique des documents juridiques
- Maintient le contexte de chaque section

#### 3. Découpage Optimisé
- **RecursiveCharacterTextSplitter** : Crée des chunks de 512 tokens avec un chevauchement de 64 tokens
- Le chevauchement assure qu'aucune information n'est perdue entre deux chunks
- Taille optimisée pour les modèles d'embeddings

#### 4. Vectorisation et Stockage
- **HuggingFaceEmbeddings** : Convertit chaque chunk de texte en un vecteur numérique
- **ChromaDB** : Stocke les vecteurs dans une base de données pour une recherche rapide
- Utilise la similarité cosinus pour trouver les passages les plus pertinents

### Variables importantes :
- `doc_splits` : Liste des chunks de documents prêts à être vectorisés
- `embeddings` : Modèle pour convertir le texte en vecteurs
- `db` : Base de données vectorielle ChromaDB

In [3]:
# Cell 3: Document Loading, Splitting, and Vectorization
# This cell handles the preprocessing of text data for retrieval.

# Import necessary libraries for document loading, text splitting, and embeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpointEmbeddings  # preferred
from langchain_community.vectorstores import Chroma
from IPython.display import display, Markdown
# 1) Load markdown files from the 'data' directory
loader = DirectoryLoader(
    path=DATA_DIR,
    loader_cls=UnstructuredMarkdownLoader,
    loader_kwargs={"mode": "single"}  # keep file as one Document
)
raw_docs = loader.load()
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
    strip_headers=False
)
docs = []
for d in raw_docs:  # loaded with mode="single"
    docs.extend(md_splitter.split_text(d.page_content))
# 3) Make final, token-aware chunks
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=512, chunk_overlap=64
)
doc_splits = splitter.split_documents(docs)

# Set Hugging Face API key for embedding models
hf_key = os.getenv("HUGGINGFACEHUB_API_KEY", "")
os.environ["HUGGINGFACEHUB_API_KEY"] = hf_key
if not hf_key:
    print("Warning: HUGGINGFACEHUB_API_KEY is empty. Set it in your environment or in a .env file.")
# 3) Initialize embeddings model to convert text chunks into vectors
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,  # Use model from config
    model_kwargs={'device': 'cpu'},  # Use CPU for computation
    # Normalize for cosine similarity
    encode_kwargs={'normalize_embeddings': True}
)

# 4) Create and persist a Chroma vector database from the document splits
db = Chroma.from_documents(
    documents=doc_splits,                 # The processed document chunks
    embedding=embeddings,                 # The embedding function to use
    collection_name=COLLECTION_NAME,      # Name from config
    # persist_directory=PERSIST_DIR,        # Directory from config
)

C:\Users\Amine's PC\AppData\Local\Temp\ipykernel_17556\780308056.py:35: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


## 🗄️ Étape 4 : Initialisation des Bases de Données

### Objectif
Établir les connexions avec les bases de données Neo4j (graphe) et ChromaDB (vecteurs), et préparer les outils de recherche.

### Ce qui se passe ici :

#### 1. Configuration du Retriever
- **MMR (Maximum Marginal Relevance)** : Stratégie de recherche qui équilibre pertinence et diversité
- `k=4` : Retourne 4 documents les plus pertinents
- `fetch_k=50` : Examine 50 candidats avant de sélectionner les 4 meilleurs
- `lambda_mult=0.35` : Balance entre similarité (pertinence) et diversité

#### 2. Connexion à Neo4j
- **Neo4jGraph** : Établit la connexion à la base de données graphe
- Stocke les relations entre documents (modifications, citations, visas)
- Permet d'exécuter des requêtes Cypher pour extraire des informations structurées

#### 3. Schema Enrichi
- **enhanced_graph** : Version améliorée avec informations détaillées sur la structure du graphe
- Affiche le schéma complet : nœuds, relations, propriétés
- Utile pour comprendre la structure des données et écrire des requêtes Cypher

### Variables importantes :
- `retriever` : Outil de recherche dans la base vectorielle
- `graph` : Connexion à Neo4j pour les requêtes graphiques
- `enhanced_graph` : Connexion avec schéma détaillé

In [ ]:
# Cell 4: Database Initialization
# This cell initializes the connection to the Neo4j graph database and prepares for GraphRAG operations.

# Import necessary libraries for graph operations and document handling
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document
from langchain_experimental.llms.ollama_functions import OllamaFunctions
from langchain_experimental.graph_transformers.diffbot import DiffbotGraphTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import Markdown
import os

# Initialize a retriever from the Chroma vector database for Maximum Marginal Relevance (MMR) search
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 50, "lambda_mult": 0.35}
)


# Access them safely
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
# Initialize a connection to the default Neo4j graph database
graph = Neo4jGraph(

)


# Initialize another connection to the Neo4j graph, this time with an enhanced schema
# The enhanced schema provides more detailed information about the graph structure
enhanced_graph = Neo4jGraph(

    enhanced_schema=True
)
# Display the enhanced graph schema as markdown for inspection
display(Markdown(enhanced_graph.schema))

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Node properties:
- **doc**
  - `doc_name`: STRING Example: "constitution_2008"
  - `doc_data`: STRING Example: "2008"
- **article**
  - `art`: STRING Example: "Art 1 - ORDONNANCE n° 03-11 — relative à la monnai"
- **visa**
  - `visa`: STRING Example: "constitution_1996"
Relationship properties:

The relationships:
(:doc)-[:cite]->(:visa)
(:doc)-[:contient]->(:article)
(:doc)-[:a_modifié]->(:doc)
(:doc)-[:constitution_modifé]->(:doc)

## 🔗 Étape 5 : Chaîne RAG Vectorielle

### Objectif
Créer une chaîne de traitement pour répondre aux questions en utilisant uniquement la base de données vectorielle (ChromaDB).

### Ce qui se passe ici :

#### 1. Configuration du LLM
- **ChatGroq** : Utilisation du modèle `llama-3.1-8b-instant` (rapide et efficace)
- Modèle optimisé pour des réponses concises et précises

#### 2. Définition du Prompt
Le prompt guide le modèle pour :
- Répondre uniquement en se basant sur le contexte fourni
- Ne pas inventer d'informations
- Garder les réponses concises (3 phrases maximum)
- Utiliser le format Markdown

#### 3. Construction de la Chaîne
- **Prompt** → **LLM** → **OutputParser** : Pipeline de traitement
- Le retriever cherche les documents pertinents
- Le LLM génère la réponse à partir du contexte
- L'OutputParser formate la réponse en texte

### Mode d'utilisation :
Cette chaîne est idéale pour des questions sur le **contenu** des documents :
- "Que dit la constitution de 2020 ?"
- "Quels sont les articles sur la liberté d'expression ?"

In [34]:
# Cell 5: Define Vector-Only RAG Chain
# This cell sets up a Retrieval-Augmented Generation (RAG) chain to answer questions using only the vector database.

from langchain.prompts import PromptTemplate
from langchain import hub
from langchain_groq import ChatGroq
import os
from langchain_core.output_parsers import StrOutputParser

# Ensure GROQ API key is available to prevent empty Bearer header issues
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError(
        "GROQ_API_KEY is not set. Add it to your environment or .env file, then re-run Cell 1 before continuing."
    )

rag_llm = ChatGroq(
    model_name=VECTOR_RAG_LLM,  # From config
)

# Define a prompt template for the RAG chain
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the question. If you don't know the answer, just do not return anything. 
    use markdown .
    Use three sentences maximum and keep the answer concise:
    Question: {question} 
    Context: {context} 
    Answer: 
    """,
    input_variables=["question", "context"],
)

# Create the RAG chain by piping the prompt, LLM, and output parser
rag_chain = prompt | rag_llm | StrOutputParser()

print("Vector-only RAG chain defined.")

Vector-only RAG chain defined.


## 📊 Étape 6 : Chaîne RAG Graphique

### Objectif
Créer une chaîne de traitement pour répondre aux questions en utilisant uniquement la base de données graphe Neo4j avec des requêtes Cypher.

### Ce qui se passe ici :

#### 1. Prompt Cypher
- **cypher_prompt** : Guide le LLM pour générer des requêtes Cypher valides
- **Règles strictes** :
  - Utiliser uniquement les labels, relations et propriétés du schéma
  - Ne pas inventer de noms ou de structures
  - Utiliser `toLower()` et `CONTAINS` pour la recherche insensible à la casse
  - Lecture seule (pas de `CREATE`, `MERGE`, `DELETE`)
  - Limiter les résultats à 50 pour des performances optimales

#### 2. Prompt de Réponse
- **qa_prompt** : Transforme les résultats Cypher en réponses en langage naturel
- Synthétise les informations structurées du graphe
- Formate la réponse en Markdown

#### 3. GraphCypherQAChain
- **Cypher LLM** : Génère la requête Cypher à partir de la question
- **QA LLM** : Formule la réponse finale à partir des résultats
- **Validation** : Vérifie la syntaxe des requêtes Cypher avant exécution
- **Return Direct** : Retourne directement les résultats du graphe

### Mode d'utilisation :
Cette chaîne est idéale pour des questions sur les **relations** entre documents :
- "Quels documents ont modifié la constitution ?"
- "Quels textes citent la LOI n° 84-17 ?"
- "Quels décrets ont été publiés à une date donnée ?"

In [35]:
# Cell 6: Define Graph-Only RAG Chain (GraphCypherQAChain)
# This cell sets up a GraphCypherQAChain to generate and execute Cypher queries against the Neo4j graph.

from langchain.prompts import PromptTemplate
from langchain.chains import GraphCypherQAChain
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI


cypher_prompt = PromptTemplate(
    template="""You are an expert at generating Cypher queries for Neo4j.


STRICT RULES:
1. **Use ONLY the labels, relationships, and properties defined in the schema above.** Do not invent or assume any others.
2. **Break down complex questions into comprehensive queries that return ALL relevant information.**
3. **DO NOT use parameters (like $name, $value, etc.). Embed values directly in the query.**
4. Use `toLower()` and `CONTAINS` for case-insensitive string matching with literal values.
5. **Never use `MERGE`, `CREATE`, `SET`, or `DELETE`.** This is a read-only system.
6. The label for a country is `contry`. The label for a person is `charachter`. Be precise.
7. Always include `LIMIT 50` to get comprehensive results.
8. **For questions about people at companies, always return: name, profession, age, origin, company name, company field.**
9. do not generate anything except cypher queries, even titles like **Cypher Query** and ```cypher and comments
10. review the cypher query before answering
11. use jsut the provided nodes's names and labels

SCHEMA:
{schema}

Question: {question}

Cypher Query:""",
    input_variables=["schema", "question"],
)

qa_prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks. 
    Use the following Cypher query results to answer the question. If you don't know the answer, just say that you don't know.
    Use markdown for the response.
    Use three sentences maximum and keep the answer concise.
    Question: {question} 
    Cypher Query: {query}
    Query Results: {context} 
    
    Answer:""",
    input_variables=["question", "query", "context"],
)

graph_llm = ChatGroq(
    model_name=GRAPH_RAG_LLM,  # From config
    temperature=0,
    max_tokens=512,
)

graph_rag_chain = GraphCypherQAChain.from_llm(
    cypher_llm=graph_llm,
    qa_llm=graph_llm,
    validate_cypher=True,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    return_direct=True,
    cypher_prompt=cypher_prompt,
    qa_prompt=qa_prompt,
    allow_dangerous_requests=True,
)

print("Graph-only RAG chain defined.")

Graph-only RAG chain defined.


## 🔄 Étape 7 : Chaîne RAG Combinée

### Objectif
Créer une chaîne de traitement hybride qui exploite à la fois la base vectorielle et la base graphe pour fournir des réponses complètes et contextualisées.

### Ce qui se passe ici :

#### 1. Architecture Hybride
Cette chaîne combine intelligemment :
- **RAG Vectorielle** : Pour le contenu et la sémantique des documents
- **RAG Graphique** : Pour les relations et la structure entre documents

#### 2. Prompt Intelligent
Le prompt guide le système pour :
- Analyser si la question nécessite des informations vectorielles, graphiques, ou les deux
- Utiliser automatiquement la source d'information la plus appropriée
- Combiner les résultats des deux sources quand nécessaire
- Fournir des réponses enrichies avec le contexte relationnel

#### 3. Logique de Routage
- **Questions de contenu** → RAG vectorielle privilégiée
- **Questions de relations** → RAG graphique privilégiée  
- **Questions complexes** → Combinaison des deux approches

### Avantages :
- **Réponses plus complètes** : Combine contenu et structure
- **Contexte enrichi** : Identifie les relations entre documents
- **Flexibilité** : S'adapte automatiquement au type de question
- **Précision accrue** : Validation croisée entre les sources

### Mode d'utilisation :
Cette chaîne est idéale pour des questions **complexes** nécessitant plusieurs types d'informations :
- "Quelles modifications ont été apportées aux articles sur la liberté d'expression ?"
- "Quel est le contenu de l'article 41 et quels documents l'ont modifié ?"

In [36]:
# Cell 7: Define Combined Graph + Vector RAG Chain
from pprint import pprint
from langchain.prompts import PromptTemplate
from groq import Groq
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# Prompt for the final answer generation


final_prompt = PromptTemplate(
    template="""You are an intelligent assistant that synthesizes information from multiple sources to provide comprehensive answers.

You have access to two types of information:
- Document Context: Detailed narrative content from documents
- Graph Context: Structured facts and relationships from a knowledge graph

Your task: Analyze both contexts and create the most complete answer possible.

Guidelines:
- If both contexts contain relevant information, combine them seamlessly
- If only one context has useful information, rely on that source
- If information conflicts between contexts, acknowledge this and explain both perspectives
- If neither context fully answers the question, clearly state what's missing

Question:
{question}

Document Context:
<<<
{context}
>>>

Graph Context:
<<<
{graph_knowledge}
>>>

Provide your response in exactly this format:

**Answer:**
[Write a complete, natural answer to the question. Do not mention sources, contexts, or data origins in this section. Focus solely on answering the question clearly and comprehensively.]

**Source:** [One concise line indicating which context(s) provided the information - choose from: "Based on document context", "Based on graph context", "Based on both document and graph contexts", or "Based on partial information from available contexts"]

Answer:""",
    input_variables=["question", "context", "graph_knowledge"],
)


# LLM for the final answer generation
final_llm = ChatGroq(
    # model_name=COMBINED_RAG_LLM,
    model_name=GRAPH_RAG_LLM,
)

# Combined chain
graph_db_rag_chain = final_prompt | final_llm | StrOutputParser()
client = Groq()


def query_graph_db(question: str) -> str:
    """Queries the graph database and returns a structured string response."""
    generation = graph_rag_chain.invoke({"query": question})['result']
    graph_response = ''
    for i in generation:
        y = ''
        for key, val in zip(i.keys(), i.values()):
            y += f'The {key} is: {val},'
        graph_response += y+'.\n\n'
    return graph_response


def summarize_graph_results(question: str, graph_data: str) -> str:
    """Uses an LLM to convert structured graph data into a natural language sentence."""
    completion = client.chat.completions.create(
        model=GRAPH_RESPONSE_LLM,
        messages=[
            {
                "role": "system",
                "content": '''You will be given a user's question and structured data retrieved from a graph. 
                you have to generate an answer for the question, the answer will start with the problem 
                and then you will use and use all all the structured data to produce a answer
                do not add any information from your knowleadge
                always use the provided structured data
                If you do not know the answer, then you have just to respone with all the structuered informations and do not say that you did not find somethig or it is unavailable, just the pure informative answer'''
            },
            {
                "role": "user",
                "content": f"The question is: {question}\nThe data is: {graph_data}"
            }
        ]
    )
    return completion.choices[0].message.content


def query_combined_dbs(question: str) -> str:
    """
    Orchestrates a query across the graph and vector databases to generate a comprehensive answer.
    """
    # 1. Query the graph and get structured data
    graph_data = query_graph_db(question)
    if not graph_data:
        print("No results from graph database. Falling back to vector-only search.")
        # Fallback to vector-only search if needed
        docs = retriever.invoke(question)
        return rag_chain.invoke({"context": docs, "question": question})

    # 2. Summarize graph data into a natural language query for the vector DB
    natural_language_graph_response = summarize_graph_results(
        question, graph_data)
    print(f"Graph Summary: {natural_language_graph_response}\n")

    # 3. Use the summary to retrieve relevant documents from the vector DB
    retrieved_docs = retriever.invoke(natural_language_graph_response)
    print(f"relevent docs: {len(retrieved_docs)}\n")
    print(
        f"relevent docs: {len(''.join([retrieved_docs[i].page_content for i in range(2)]))}\n")
    print(f"relevent docs: {retrieved_docs}\n")

    # 4. Generate the final answer using the original question, graph summary, and vector context
    final_answer = graph_db_rag_chain.invoke({
        "context": retrieved_docs,
        "question": question,
        'graph_knowledge': natural_language_graph_response
    })
    return final_answer


print("Combined RAG chain and helper functions defined.")

Combined RAG chain and helper functions defined.


---

# 🧪 Démonstrations Pratiques : 10 Exemples d'Usage

Cette section présente 10 exemples concrets d'utilisation des différentes chaînes RAG pour répondre à des questions variées sur les documents juridiques algériens. Chaque exemple illustre une approche spécifique et ses avantages.

## 📊 Types de Requêtes Testées

### 🔗 RAG Vectorielle (Questions de Contenu)
- **Objectif** : Rechercher des informations dans le contenu des documents
- **Cas d'usage** : Comprendre le contenu d'un texte juridique spécifique

### 📊 RAG Graphique (Questions Relationnelles) 
- **Objectif** : Explorer les relations entre documents
- **Cas d'usage** : Identifier qui modifie quoi, quelles sont les citations, etc.

### 🔄 RAG Combinée (Questions Complexes)
- **Objectif** : Combiner contenu et relations pour des réponses complètes
- **Cas d'usage** : Questions nécessitant à la fois du contenu et du contexte relationnel

---

## 🔗 Exemple 1 : RAG Vectorielle - Question de Contenu Simple

### 🎯 Objectif de cette démonstration
Tester la capacité du système à **récupérer et synthétiser le contenu** d'un document juridique spécifique.

### 📋 Type de Question
**Question de contenu direct** : Recherche d'informations générales sur un document précis (Constitution de 2020).

### 🔧 Méthodologie
1. **RAG Vectorielle uniquement** : Utilise ChromaDB pour la recherche sémantique
2. **Retriever** : Trouve les 4 passages les plus pertinents 
3. **LLM** : Synthétise les informations trouvées

### 💡 Cas d'usage attendu
Ce type de question est idéal quand l'utilisateur veut connaître le **contenu général** d'un texte juridique sans s'intéresser aux relations avec d'autres documents.

---

In [ ]:
# Cell 8: Demonstrations
from IPython.display import Markdown

# --- Example 1: Vector-Only Query ---
question_1 = "Que savez-vous de la constitution de 2020 ?"
print(f"--- Running Vector-Only Query ---\nQuestion: {question_1}")
docs = retriever.invoke(question_1)
generation = rag_chain.invoke({"context": docs, "question": question_1})
display(Markdown(generation))

--- Running Vector-Only Query ---
Question: what do u know about que ceque vous savez sur la constituion 2020


### Constitutions 2020
#### Résumé
La Constitution de la République algérienne démocratique et populaire a été promulguée en 2020. Elle fixe les principes généraux de la société algérienne, notamment l'unité et l'indivisibilité de l'Algérie, l'Islam comme religion de l'État, l'Arabe comme langue nationale et officielle, et Tamazight également langue nationale et officielle.

#### Articles clés
- Article 1er : L'Algérie est une République Démocratique et Populaire, indivisible.
- Article 2 : L'Islam est la religion de l'État.
- Article 3 : L'Arabe est la langue nationale et officielle.
- Article 4 : Tamazight est également langue nationale et officielle.

## 📊 Exemple 2 : RAG Graphique - Question Relationnelle Pure

### 🎯 Objectif de cette démonstration
Tester la capacité du système à **identifier les relations de modification** entre documents juridiques.

### 📋 Type de Question
**Question relationnelle** : Recherche de documents qui ont des relations spécifiques (modifications de la constitution).

### 🔧 Méthodologie
1. **RAG Graphique uniquement** : Utilise Neo4j avec des requêtes Cypher
2. **Cypher LLM** : Génère une requête pour trouver les relations "MODIFIE"
3. **QA LLM** : Transforme les résultats structurés en langage naturel

### 💡 Cas d'usage attendu
Parfait pour des questions de **traçabilité juridique** : qui modifie quoi, quand, et comment les documents sont liés entre eux.

---

In [64]:
# --- Example 2: Graph-Only Query ---
question_2 = "Quels sont les documents qui ont modifié la constitution nationale ?"
print(f"\n--- Running Graph-Only Query ---\nQuestion: {question_2}")
graph_response = query_graph_db(question_2)
natural_graph_response = summarize_graph_results(question_2, graph_response)
display(Markdown(natural_graph_response))


--- Running Graph-Only Query ---
Question:  quelles sont les docuemtn qui ont modifié la constitution national


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc)-[:constitution_modifé]->(c:doc)
RETURN d.doc_name, d.doc_data
LIMIT 50

> Finished chain.


**Problème :** Quelles sont les documents qui ont modifié la Constitution nationale ?  

- **Document :** constitution_2002 – **Date :** 2002  
- **Document :** constitution_2008 – **Date :** 2008  
- **Document :** constitution_2016 – **Date :** 2016  
- **Document :** constitution_2020 – **Date :** 2020

## 🔄 Exemple 3 : RAG Combinée - Analyse Comparative Complexe

### 🎯 Objectif de cette démonstration
Tester la capacité du système à **combiner informations relationnelles et contenu** pour faire une analyse comparative.

### 📋 Type de Question
**Question complexe combinée** : Nécessite d'identifier les documents (graphe) ET de comparer leur contenu (vectoriel).

### 🔧 Méthodologie
1. **RAG Graphique** : Identifie d'abord quels documents ont modifié la constitution
2. **RAG Vectorielle** : Récupère le contenu de ces documents
3. **Synthèse intelligente** : Compare et analyse les différences de contenu

### 💡 Cas d'usage attendu
Questions d'**analyse juridique avancée** nécessitant à la fois la traçabilité des relations et l'analyse du contenu pour fournir des insights complets.

---

In [65]:
# --- Example 3: Combined Query ---
question_3 = "Comparer le contenu des documents qui ont modifié la constitution nationale."
print(f"\n--- Running Combined Query ---\nQuestion: {question_3}")
display(Markdown(query_combined_dbs(question=question_3)))


--- Running Combined Query ---
Question:  comparer le contenu des docuemtn qui ont modifié la constitution national


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc)-[:constitution_modifé]->(:doc)
RETURN d.doc_name AS name, d.doc_data AS content
LIMIT 50

> Finished chain.
Graph Summary: **Problème :** comparer le contenu des documents qui ont modifié la Constitution nationale.

**Données fournies :**

- **Document :** constitution_2002  
  **Contenu :** 2002

- **Document :** constitution_2008  
  **Contenu :** 2008

- **Document :** constitution_2016  
  **Contenu :** 2016

- **Document :** constitution_2020  
  **Contenu :** 2020

**Comparaison :**  
Chaque document porte le nom d’une année correspondant à la révision de la Constitution nationale et son contenu se limite à l’année même (2002, 2008, 2016, 2020). Ainsi, le contenu diffère d’un document à l’autre uniquement par l’année de modification indiquée.

relevent docs: 4

relevent docs: 3592

releven

**Answer:**
Les documents qui ont modifié la Constitution nationale se distinguent principalement par les années de révision : 2002, 2008, 2016 et 2020.  

- **Révision de 2002** : le texte de la loi n° 02‑03 porte une modification concrète : il ajoute l’article 3 bis qui reconnaît le tamazight comme langue nationale et prévoit que l’État doit promouvoir et développer toutes ses variétés linguistiques sur le territoire. Cette modification est explicitement détaillée dans le document législatif.  

- **Révisions de 2008, 2016 et 2020** : les informations disponibles se limitent à l’indication de l’année de la révision (respectivement 2008, 2016 et 2020). Aucun détail sur le contenu précis de ces révisions n’est fourni dans les documents consultés ; on ne sait donc pas quelles dispositions constitutionnelles ont été modifiées ou ajoutées à ces dates.  

En résumé, le seul texte dont le contenu est clairement décrit est celui de 2002, qui introduit la reconnaissance du tamazight. Pour les révisions de 2008, 2016 et 2020, le contenu reste inconnu dans les sources actuelles, se résumant à la simple mention de l’année de la modification.  

**Source:** Based on both document and graph contexts.

## 📊 Exemple 4 : RAG Graphique - Recherche de Modifications Spécifiques

### 🎯 Objectif de cette démonstration
Tester la recherche précise de **documents qui modifient un texte juridique spécifique** (LOI n° 84-17).

### 📋 Type de Question
**Question relationnelle ciblée** : Identification des relations de modification pour un document nommément désigné.

### 🔧 Méthodologie
1. **Recherche Cypher précise** : Utilise `CONTAINS` et `toLower()` pour trouver le document exact
2. **Relations MODIFIE** : Cherche tous les documents qui ont des liens de modification
3. **Résultats structurés** : Liste ordonnée des documents modificateurs

### 💡 Cas d'usage attendu
**Traçabilité juridique précise** : Quand un juriste doit connaître tous les amendements et modifications apportés à une loi spécifique.

---

In [ ]:
# --- Example 4: Graph-Only Query ---
question_4 = "Quels sont les documents qui ont modifié le document : 'LOI n° 84-17 — relative aux lois de finances' ?"
print(f"\n--- Running Graph-Only Query ---\nQuestion: {question_4}")
graph_response = query_graph_db(question_4)
natural_graph_response = summarize_graph_results(question_4, graph_response)
display(Markdown(natural_graph_response))


--- Running Graph-Only Query ---
Question: quelle sont les document qui ont modifié le docuemnt:  'LOI n° 84-17 — relative aux lois de finances'


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (target:doc)
WHERE toLower(target.doc_name) CONTAINS toLower('LOI n° 84-17 — relative aux lois de finances')
MATCH (modifier:doc)-[:a_modifié]->(target)
RETURN modifier.doc_name, modifier.doc_data
LIMIT 50

> Finished chain.


**Problème :** identifier les documents qui ont modifié le texte : « LOI n° 84-17 — relative aux lois de finances ».

**Documents modificateurs :**

- DECRET PRESIDENTIEL n° 21-478 — autorisant la participation de l'Algérie à la 6ᵉ augmentation générale du capital de la Banque islamique de développement. (date : 2021‑11‑29)  
- DECRET EXECUTIF n° 21-473 — modifiant la répartition par secteur des dépenses d'équipement de l'État pour 2021. (date : 2021‑11‑25)  
- DECRET EXECUTIF n° 21-474 — portant virement de crédits au sein du budget de fonctionnement du ministère de l'énergie et des mines. (date : 2021‑11‑25)  
- DECRET EXECUTIF n° 21-475 — portant virement de crédits au sein du budget de fonctionnement du ministère de la formation et de l'enseignement professionnels. (date : 2021‑11‑25)  
- DECRET EXECUTIF n° 21-476 — portant virement de crédits au sein du budget de fonctionnement du ministère de l'habitat, de l'urbanisme et de la ville. (date : 2021‑11‑25)  
- DECRET EXECUTIF n° 09-313 — portant création, organisation et fonctionnement de l'institut algérien des mines. (date : aucune date disponible)  
- DECRET PRESIDENTIEL n° 12-413 — portant transfert de crédits au budget de fonctionnement du ministère de l'intérieur et des collectivités locales. (date : aucune date disponible)  
- DECRET PRESIDENTIEL n° 12-414 — portant transfert de crédits au budget de fonctionnement du ministère des affaires étrangères. (date : aucune date disponible)  
- DECRET EXECUTIF n° 12-421 — modifiant la répartition par secteur des dépenses d'équipement de l'État pour 2012. (date : aucune date disponible)  
- DECRET EXECUTIF n° 12-422 — portant virement de crédits au sein du budget de fonctionnement du ministère de la justice. (date : aucune date disponible)  

## 📊 Exemple 5 : RAG Graphique - Analyse des Citations (Visa)

### 🎯 Objectif de cette démonstration
Tester la capacité à identifier les **relations de citation** entre documents juridiques (visas constitutionnels).

### 📋 Type de Question
**Question de référencement juridique** : Recherche de documents qui citent la constitution comme base légale (visa).

### 🔧 Méthodologie
1. **Recherche de relations CITE** : Utilise les liens de citation dans le graphe
2. **Filtrage par type** : Cherche spécifiquement les citations de constitution
3. **Classification** : Distingue les visas des autres types de références

### 💡 Cas d'usage attendu
**Analyse de la hiérarchie juridique** : Comprendre quels textes s'appuient sur la constitution et comment l'autorité constitutionnelle est invoquée.

---

In [99]:
# --- Example 5: Graph-Only Query ---
question_5 = "Quels documents citent des constitution en visa ?"
print(f"\n--- Running Graph-Only Query ---\nQuestion: {question_5}")
graph_response = query_graph_db(question_5)
natural_graph_response = summarize_graph_results(question_5, graph_response)

display(Markdown(natural_graph_response))


--- Running Graph-Only Query ---
Question: Quels documents citent des constitution en visa ?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc)-[:cite]->(v:visa)
RETURN d.doc_name AS name, d.doc_data AS data
LIMIT 50
Generated Cypher:
MATCH (d:doc)-[:cite]->(v:visa)
RETURN d.doc_name AS name, d.doc_data AS data
LIMIT 50

> Finished chain.

> Finished chain.



--- Running Graph-Only Query ---
Question: Quels documents citent des constitution en visa ?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc)-[:cite]->(v:visa)
RETURN d.doc_name AS name, d.doc_data AS data
LIMIT 50
Generated Cypher:
MATCH (d:doc)-[:cite]->(v:visa)
RETURN d.doc_name AS name, d.doc_data AS data
LIMIT 50

> Finished chain.

> Finished chain.


**Problème :** Quels documents citent des constitution en visa ?

**Documents disponibles :**

- ORDONNANCE n° 03-11 — relative à la monnaie et au crédit. (26 août 2003)  
- ORDONNANCE n° 03-12 — relative à l'obligation d'assurance des catastrophes naturelles et à l'indemnisation des victimes. (26 août 2003)  
- DÉCRET PRÉSIDENTIEL n° 03-281 — portant transfert de crédits au budget de fonctionnement du ministère de l'intérieur et des collectivités locales. (24 août 2003)  
- DÉCRET PRÉSIDENTIEL n° 03-282 — portant transfert de crédits au budget de fonctionnement du ministère des affaires étrangères. (24 août 2003)  
- DÉCRET EXÉCUTIF n° 03-283 — portant création d'un chapitre et virement de crédits, au sein du budget de fonctionnement du ministère de l'intérieur et des collectivités locales. (25 août 2003)  
- DÉCRET EXÉCUTIF n° 03-284 — fixant les conditions et les modalités d'octroi d'aides au profit des familles des victimes et aux sinistrés du séisme du 21 mai 2003. (25 août 2003)  
- DECRET EXECUTIF n° 04-379 — instituant une indemnité de responsabilité personnelle au profit des agents comptables agréés auprès des postes diplomatiques et consulaires. (28 novembre 2004)  
- DECRET EXECUTIF n° 04-380 — portant virement de crédits au sein du budget de fonctionnement du ministère du commerce. (28 novembre 2004)  
- DECRET EXECUTIF n° 04-381 — fixant les règles de la circulation routière. (28 novembre 2004)  
- LOI n° 04-18 — relative à la prévention et à la répression de l'usage et du trafic illicites de stupéfiants et de substances psychotropes. (25 décembre 2004)  

## 📊 Exemple 6 : RAG Graphique - Recherche Temporelle

### 🎯 Objectif de cette démonstration
Tester la capacité à effectuer des **recherches par date de publication** pour identifier les textes publiés à un moment précis.

### 📋 Type de Question
**Question chronologique** : Recherche de décrets exécutifs publiés à une date spécifique (2021-11-25).

### 🔧 Méthodologie
1. **Filtrage temporel** : Utilise les propriétés de date dans Neo4j
2. **Type de document** : Filtre par type "DECRET EXECUTIF"
3. **Résultats datés** : Liste les documents avec leur date de publication

### 💡 Cas d'usage attendu
**Veille juridique temporelle** : Suivre l'activité réglementaire à des moments clés, analyser les pics d'activité législative.

---

In [70]:
# --- Example 6: Graph-Only Query ---
question_6 = "Donnez-moi quelques décrets exécutifs qui ont été publiés le 2021-11-25."
print(f"\n--- Running Graph-Only Query ---\nQuestion: {question_6}")
graph_response = query_graph_db(question_6)
natural_graph_response = summarize_graph_results(question_6, graph_response)

display(Markdown(natural_graph_response))


--- Running Graph-Only Query ---
Question: donne moi quelque DECRET EXECUTIF qui sont publié dans 2021‑11‑25 


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc)
WHERE toLower(d.doc_name) CONTAINS toLower('DECRET EXECUTIF')
  AND d.doc_data CONTAINS '2021-11-25'
RETURN d.doc_name, d.doc_data
LIMIT 50

> Finished chain.


**Problème :** Trouver les décrets exécutifs publiés le 25 novembre 2021.

**Réponse :**  
- **DECRET EXECUTIF n° 21-473** — modifiant la répartition par secteur des dépenses d'équipement de l'Etat pour 2021. (date de publication : 2021‑11‑25)  
- **DECRET EXECUTIF n° 21-474** — portant virement de crédits au sein du budget de fonctionnement du ministère de l'énergie et des mines. (date de publication : 2021‑11‑25)  
- **DECRET EXECUTIF n° 21-475** — portant virement de crédits au sein du budget de fonctionnement du ministère de la formation et de l'enseignement professionnels. (date de publication : 2021‑11‑25)  
- **DECRET EXECUTIF n° 21-476** — portant virement de crédits au sein du budget de fonctionnement du ministère de l'habitat, de l'urbanisme et de la ville. (date de publication : 2021‑11‑25)

## 🔄 Exemple 7 : RAG Combinée - Analyse de Modifications avec Contenu

### 🎯 Objectif de cette démonstration
Tester la capacité à **identifier les modifications ET analyser leur contenu** pour un décret exécutif spécifique.

### 📋 Type de Question
**Question hybride complexe** : Nécessite à la fois l'identification des relations (qui modifie) et l'analyse du contenu (quelles nouveautés).

### 🔧 Méthodologie
1. **Phase Graphique** : Trouve les documents qui ont modifié le DÉCRET EXÉCUTIF n° 98-227
2. **Phase Vectorielle** : Analyse le contenu de ces documents modificateurs
3. **Synthèse** : Identifie les nouveautés et changements apportés

### 💡 Cas d'usage attendu
**Analyse juridique complète** : Comprendre non seulement qui a modifié un texte, mais aussi quelles sont les implications concrètes de ces modifications.

---

In [79]:
# --- Example 7: Combined Query ---
question_7 = "Donnez-moi 3 documents qui ont modifié le document 'DECRET EXECUTIF n° 98-227', ainsi que les nouveautés qui ont été ajoutées à ce document."
print(f"\n--- Running Combined Query ---\nQuestion: {question_7}")
display(Markdown(query_combined_dbs(question=question_7)))


--- Running Combined Query ---
Question: donne moi 3 documents qui ont modifié le document 'DECRET EXECUTIF n° 98-227', et le nouveau trucs qui sont ajouté à ce document


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (mod:doc)-[:a_modifié]->(target:doc {doc_name: 'DECRET EXECUTIF n° 98-227'})
RETURN mod.doc_name AS modifyingDocument, mod.doc_data AS newContent
LIMIT 50

> Finished chain.
No results from graph database. Falling back to vector-only search.


### Documents qui ont modifié le "DECRET EXECUTIF n° 98-227"
#### 1. **DECRET EXECUTIF n° 24-102**
Modifie et complète le décret exécutif n° 08-129 du 27 Rabie Ethani 1429 correspondant au 3 mai 2008 portant statut particulier de l'enseignant chercheur hospitalo-universitaire.

#### 2. **DECRET EXECUTIF n° 24-284**
Modifie et complète le décret exécutif n° 20-339 du 6 Rabie Ethani 1442 correspondant au 22 novembre 2020 portant création de l'université de Relizane.

#### 3. **DECRET EXECUTIF n° 97-53**
Complété, insère un article 3 bis 16 qui précise les dispositions du présent décret en tant que de besoin.

### Nouveaux articles ajoutés
* Article 3 bis 16 : Les dispositions du présent décret sont précisées en tant que de besoin, par un arrêté conjoint du ministre chargé du commerce et du ministre chargé des finances.
* Article 2 : Le règlement intérieur-type prévu à l'article 1er ci-dessus, fixe les règles d'organisation et de fonctionnement du comité de wilaya, ainsi que les droits et les obligations de ses membres.
* Article 3 : Le comité de wilaya est chargé, notamment : — d'étudier, de proposer et de veiller à la mise en œuvre de toutes mesures concourant à la prévention et à la lutte contre la violence dans les infrastructures sportives au niveau local.

## 🔄 Exemple 8 : RAG Combinée - Exploration Structurée de Contenu

### 🎯 Objectif de cette démonstration
Tester la capacité à **extraire et organiser le contenu structuré** d'un document juridique (articles et leurs contenus).

### 📋 Type de Question
**Question d'exploration structurée** : Recherche de l'architecture interne d'un document (articles de l'ORDONNANCE n° 03-11).

### 🔧 Méthodologie
1. **Identification** : Utilise le graphe pour confirmer l'existence du document
2. **Extraction de contenu** : Utilise la base vectorielle pour trouver les articles
3. **Organisation** : Structure la réponse selon l'organisation juridique du texte

### 💡 Cas d'usage attendu
**Analyse de structure juridique** : Comprendre l'organisation interne d'un texte, ses articles, et leur contenu pour une analyse complète.

---

In [82]:
# --- Example 8: Combined Query ---
question_8 = "Liste-moi les articles contenus dans le document 'ORDONNANCE n° 03-11' et leur contenu."
print(f"\n--- Running Combined Query ---\nQuestion: {question_8}")
display(Markdown(query_combined_dbs(question=question_8)))


--- Running Combined Query ---
Question: Liste-moi les articles contenus dans le document 'ORDONNANCE n° 03-11' et leur contenu


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc {doc_name: 'ORDONNANCE n° 03-11'})-[:contient]->(a:article)
RETURN a.art AS article, a.art AS content
LIMIT 50
Generated Cypher:
MATCH (d:doc {doc_name: 'ORDONNANCE n° 03-11'})-[:contient]->(a:article)
RETURN a.art AS article, a.art AS content
LIMIT 50

> Finished chain.
No results from graph database. Falling back to vector-only search.

> Finished chain.
No results from graph database. Falling back to vector-only search.


**Articles contenus dans le document 'ORDONNANCE n° 03-11'**

* Article 1er : Définition de l'unité monétaire de la République algérienne démocratique et populaire, le dinar algérien.
* Article 2 : La monnaie fiduciaire est constituée de billets de banque et de pièces de monnaie métallique, et le privilège d'émettre appartient à l'État, délégué à la banque centrale.
* Article 3 : L'ordonnateur concerné doit engager et ordonnancer ou mandater le montant total ou partiel de la condamnation pécuniaire dans un délai de deux mois.
* Article 4 : Dans le cas d'insuffisance de crédits budgétaires, l'engagement et l'ordonnancement ou le mandatement sont effectués dans la limite des crédits disponibles.
* Article 5 : (Incomplet)
* Article 10 : Modification des dispositions des articles 18 bis 1 et 20 bis de la loi n° 05-01 du 27 Dhou El Hidja 1425.
* Article 11 : Complétion de la loi n° 05-01 du 27 Dhou El Hidja 1425 par l'article 20 bis 1.
* Article 12 : Modification des dispositions de l'article 27 de la loi n° 05-01 du 27 Dhou El Hidja 1425.
* Article 1er du Décret : Fixation des conditions d'admission de l'étudiant étranger au sein des établissements d'enseignement et de formation supérieurs algériens.
* Article 2 du Décret : Modalités de l'admission de l'étudiant étranger.
* Article 3 du Décret : Système d'étude et de formation applicable pour l'obtention des diplômes d'enseignement supérieur algériens.
* Article 4 du Décret : Conditions d'admission pour les étudiants étrangers.

## 🔄 Exemple 9 : RAG Combinée - Analyse des Citations Constitutionnelles

### 🎯 Objectif de cette démonstration
Tester la capacité à **identifier et contextualiser les citations constitutionnelles** dans l'ensemble du corpus juridique.

### 📋 Type de Question
**Question de référencement constitutionnel** : Recherche exhaustive des documents qui citent la constitution comme visa.

### 🔧 Méthodologie
1. **Recherche graphique étendue** : Identifie tous les documents avec des relations de citation vers la constitution
2. **Enrichissement vectoriel** : Ajoute le contexte de ces citations
3. **Synthèse hiérarchique** : Organise par type de document et pertinence

### 💡 Cas d'usage attendu
**Cartographie de l'autorité constitutionnelle** : Comprendre comment et où la constitution est invoquée dans l'ordre juridique.

---

In [89]:
# --- Example 9: Combined Query ---
question_9 = "Quels documents juridiques ont cité la 'constitution' comme visa ?"
print(f"\n--- Running Combined Query ---\nQuestion: {question_9}")
display(Markdown(query_combined_dbs(question=question_9)))


--- Running Combined Query ---
Question: Quels document juridique ont cités comme visa  la 'constitution' ?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc)-[:cite]->(v:visa)
WHERE toLower(v.visa) CONTAINS toLower('constitution')
RETURN d.doc_name AS doc_name, d.doc_data AS doc_data
LIMIT 50
Generated Cypher:
MATCH (d:doc)-[:cite]->(v:visa)
WHERE toLower(v.visa) CONTAINS toLower('constitution')
RETURN d.doc_name AS doc_name, d.doc_data AS doc_data
LIMIT 50

> Finished chain.

> Finished chain.
Graph Summary: **Problème** : Identifier les documents juridiques qui ont cité la « Constitution » comme visa.

**Réponse** : Tous les documents juridiques listés ci‑dessous font référence à la Constitution comme visa (selon les informations fournies) :

| Document juridique | Date |
|---------------------|------|
| ORDONNANCE n° 03‑11 — relative à la monnaie et au crédit. | 2003‑08‑26 |
| ORDONNANCE n° 03‑12 — relative à l'obligation d'assurance des catastrophes natu

**Answer:**
Les documents juridiques qui ont cité la « Constitution » comme visa sont :

| Document juridique | Date |
|---------------------|------|
| Ordonnance n° 03‑11 — relative à la monnaie et au crédit | 26 août 2003 |
| Ordonnance n° 03‑12 — relative à l'obligation d'assurance des catastrophes naturelles et à l'indemnisation des victimes | 26 août 2003 |
| Décret présidentiel n° 03‑281 — transfert de crédits au budget de fonctionnement du ministère de l'Intérieur et des collectivités locales | 24 août 2003 |
| Décret présidentiel n° 03‑282 — transfert de crédits au budget de fonctionnement du ministère des Affaires étrangères | 24 août 2003 |
| Décret exécutif n° 03‑283 — création d'un chapitre et virement de crédits au sein du budget de fonctionnement du ministère de l'Intérieur et des collectivités locales | 25 août 2003 |
| Décret exécutif n° 03‑284 — conditions et modalités d'octroi d'aides aux familles des victimes du séisme du 21 mai 2003 | 25 août 2003 |
| Décret exécutif n° 04‑379 — indemnité de responsabilité personnelle pour les agents comptables auprès des postes diplomatiques et consulaires | 28 novembre 2004 |
| Décret exécutif n° 04‑380 — virement de crédits au sein du budget de fonctionnement du ministère du Commerce | 28 novembre 2004 |
| Décret exécutif n° 04‑381 — règles de la circulation routière | 28 novembre 2004 |
| Loi n° 04‑18 — prévention et répression de l'usage et du trafic illicites de stupéfiants et de substances psychotropes | 25 décembre 2004 |

**Source:** Based on graph context

## 🔄 Exemple 10 : RAG Combinée - Analyse Bidirectionnelle Complète

### 🎯 Objectif de cette démonstration
Tester la capacité à effectuer une **analyse bidirectionnelle complète** : quoi modifie quoi ET quel est le contenu.

### 📋 Type de Question
**Question d'analyse juridique complète** : Combine identification des relations (LOI n° 18-06 modifie quoi) et analyse de contenu (quels articles).

### 🔧 Méthodologie
1. **Analyse des relations sortantes** : Que modifie la LOI n° 18-06 ?
2. **Analyse de contenu interne** : Quels articles contient cette loi ?
3. **Synthèse bidirectionnelle** : Impact et contenu de la loi

### 💡 Cas d'usage attendu
**Analyse juridique experte** : Vue d'ensemble complète d'un texte juridique : son impact sur d'autres textes et son contenu propre.

---

In [93]:
# --- Example 10: Combined Query ---
question_10 = "Le document 'LOI n° 18-06' a modifié quel document, et quels articles contient-il ?"
print(f"\n--- Running Combined Query ---\nQuestion: {question_10}")
display(Markdown(query_combined_dbs(question=question_10)))


--- Running Combined Query ---
Question: Le document 'LOI n° 18-06' a modifié quel document, et quels articles contient-il ?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (d:doc) 
WHERE toLower(d.doc_name) CONTAINS toLower('LOI n° 18-06')
OPTIONAL MATCH (d)-[:a_modifié]->(mod:doc)
OPTIONAL MATCH (d)-[:contient]->(a:article)
RETURN d.doc_name AS sourceDocument, 
       mod.doc_name AS modifiedDocument, 
       collect(a.art) AS articles
LIMIT 50
Generated Cypher:
MATCH (d:doc) 
WHERE toLower(d.doc_name) CONTAINS toLower('LOI n° 18-06')
OPTIONAL MATCH (d)-[:a_modifié]->(mod:doc)
OPTIONAL MATCH (d)-[:contient]->(a:article)
RETURN d.doc_name AS sourceDocument, 
       mod.doc_name AS modifiedDocument, 
       collect(a.art) AS articles
LIMIT 50

> Finished chain.

> Finished chain.
Graph Summary: **Problème**  
Quel document a été modifié par la « LOI n° 18‑06 » et quels articles cette loi comprend‑elle ?

**Réponse**  
La « LOI n° 18‑06 » modifie et complète **l’or

**Answer:**  
La LOI n° 18‑06 du 25 Ramadhan 1439 (10 juin 2018) a pour objet de **modifier et de compléter l’ordonnance n° 66‑155 du 8 juin 1966 portant code de procédure pénale**.  

Elle est structurée en plusieurs articles numérotés, allant au moins de **l’article 1 à l’article 9** :  

- **Art. 1** – indique que la loi a pour but de modifier et compléter l’ordonnance n° 66‑155 du 8 juin 1966.  
- **Art. 2** – précise les articles de l’ordonnance qui sont modifiés, complétés et rédigés de nouveau (parmi eux : 599, 602, 603, 609, 618, 620, 624, 626, 627, 628, 629 et 630).  
- **Art. 3** – (non détaillé dans les extraits, mais fait partie de la numérotation).  
- **Art. 4** – modifie et complète d’autres articles de l’ordonnance, notamment les articles 632, 633, 646, 647, 648, 649, 650, 651, 653, 654, 655, 656, 657, 658, 664, 665, 666, 667, 668, 669, 674 et 675.  
- **Art. 5 à Art. 9** – (mentionnés dans le graphe comme parties de la loi, poursuivant les dispositions de modification et de complément du code de procédure pénale).

Ainsi, la LOI n° 18‑06 modifie l’ordonnance n° 66‑155 du 8 juin 1966 et comprend au moins les articles 1 à 9, dont les premiers détaillent l’objet de la loi et les articles de l’ordonnance qui sont révisés.  

**Source:** Based on both document and graph contexts.